# Cost-Optimal AI Evaluation: Getting the Most From Your Annotation Budget

**Cost-Optimal Random Sampling** addresses the question every practitioner faces when evaluating AI systems at scale: given that LLM labels are cheap but biased, and human labels are accurate but expensive, what fraction of conversations should be sent to a human annotator to minimise estimation error within a fixed budget?

---

**What you will learn:**

- Why naively spending your annotation budget on fully-annotated samples leaves cheaper proxy labels on the table
- How the cost ratio between LLM and human labels determines the optimal annotation rate
- How GLIDE combines cheap proxy labels with expensive human annotations to produce a valid, bias-corrected metric that fits your annotation budget

## The problem: evaluation with two annotation sources at very different prices

Assume you run a customer-facing agentic assistant handling thousands of conversations per day.

### The signals

- **Every tenth user** reports incorrect or fabricated information.
- You deploy an LLM judge to measure the hallucination rate. It tells you the rate is **5%**.

Users and judge **disagree**. The latter is systematically optimistic.

### Two annotation sources, two price tags

To evaluate the hallucination rate at scale, you have access to two sources:

- **LLM judge**: $0.01 per conversation. Fast and cheap, but systematically biased.
- **Human annotator**: $1.00 per annotation. Ground-truth quality, but 100× more expensive.

### Fixing an annotation budget

You annotate 100 conversations with **both** sources upfront — a burn-in set used to calibrate the cost-optimal sampler. You then set aside **$400** for the remaining 2,100 conversations. Every dollar of that budget must cover both the LLM evaluation cost and any human annotation cost for each conversation processed.

### The key question: how many human annotations per dollar?

A naive strategy sends every processed conversation to a human annotator ($\pi = 1$). This is simple, but expensive: at $1.01 per conversation (LLM + human), a $400 budget covers only **396** conversations, leaving 1,704 conversations with no proxy labels to leverage.

The **cost-optimal random sampler** finds a better allocation: it covers **all** 2,100 conversations with a proxy label, and sends only a fraction $\pi^*$ to a human annotator — the fraction that minimises estimation variance for the given cost ratio. Prediction-powered inference then combines the cheap proxy labels with the expensive human labels to produce an unbiased, tight estimate.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ASIMeanEstimator, ClassicalMeanEstimator
from glide.samplers import CostOptimalRandomSampler
from glide.simulators import generate_binary_dataset_with_oracle_sampling

# ── Colour palette ────────────────────────────────────────────
C_JUDGE = "#E74C3C"  # LLM judge          — red-orange
C_UNIFORM = "#2E86AB"  # Uniform sampler    — blue
C_GLIDE = "#27AE60"  # GLIDE cost-optimal — green
C_TRUTH = "#2C3E50"  # True value         — dark slate

# ── Global plot style ───────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titlepad": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    }
)

## Simulating 2,200 conversations with a biased judge

`generate_binary_dataset_with_oracle_sampling` produces a synthetic dataset that mirrors the scenario above. The uncertainty score is not used here: the cost-optimal sampler requires only proxy labels and costs, not uncertainty scores.

| Array | Meaning |
|-------|--------|
| `y_true_oracle` | Ground-truth binary label: $Y_i = 1$ = hallucination confirmed by a human annotator |
| `y_proxy` | Binary label from the LLM judge: $\tilde{Y}_i = 1$ = hallucination flagged |

The first `N_BURN_IN = 100` conversations are annotated by **both** sources upfront to calibrate the cost-optimal sampler.

> **Simulated annotation:** In practice, `y_true_oracle` is revealed only after a human annotates the conversation. Here we generate all ground-truth labels upfront so that we can simulate the annotation process by masking unlabeled samples with `np.nan`.

In [ ]:
N_TOTAL = 2200
N_BURN_IN = 100  # conversations annotated by both sources upfront
N_MAIN = N_TOTAL - N_BURN_IN  # 2,100 remaining conversations
TRUE_RATE = 0.10  # true hallucination rate (unknown in practice)
COST_PROXY = 0.01  # LLM judge: $0.01 per conversation
COST_HUMAN = 1.00  # human annotator: $1.00 per annotation
BUDGET_MAIN = 400  # annotation budget for the main set ($)
RANDOM_SEED = 0

y_true_oracle, y_proxy, _ = generate_binary_dataset_with_oracle_sampling(
    n_total=N_TOTAL,
    true_mean=TRUE_RATE,
    proxy_mean=0.05,
    correlation=0.55,
    random_seed=RANDOM_SEED,
)

In [ ]:
print(f"Total conversations        : {N_TOTAL:,}")
print(f"Burn-in (calibration) set  : {N_BURN_IN}")
print(f"Main evaluation set        : {N_MAIN:,}")
print()
print(f"LLM judge cost             : ${COST_PROXY:.2f} per conversation")
print(f"Human annotator cost       : ${COST_HUMAN:.2f} per annotation")
print(f"Cost ratio (human / LLM)   : {COST_HUMAN / COST_PROXY:.0f}x")
print(f"Main annotation budget     : ${BUDGET_MAIN:.0f}")
print()
n_uniform_main = int(np.floor(BUDGET_MAIN / (COST_PROXY + COST_HUMAN)))
print(
    f"Uniform (pi=1): processes {n_uniform_main} main conversations  "
    f"at ${COST_PROXY + COST_HUMAN:.2f} each = ${n_uniform_main * (COST_PROXY + COST_HUMAN):.2f}"
)

## Step 1: Calibrate the sampler with a burn-in dataset

The cost-optimal sampler must estimate two statistics from the burn-in data before it can determine the optimal annotation rate:

- **$\text{Var}(H)$**: variance of the ground-truth labels — how spread out the true hallucination outcomes are.
- **$\text{MSE}(H, G) = \mathbb{E}[(H - G)^2]$**: mean squared error of the proxy relative to ground truth — how unreliable the LLM judge is on average.

The burn-in set is annotated by both sources at full cost ($c_g + c_h$ per sample). The estimated statistics determine $\pi^*$, the optimal probability at which to also query a human annotator for each main-set conversation.

In [ ]:
y_true_burn_in = y_true_oracle[:N_BURN_IN]
y_proxy_burn_in = y_proxy[:N_BURN_IN]

sampler = CostOptimalRandomSampler()
sampler.fit(y_true_burn_in, y_proxy_burn_in)

burn_in_cost = N_BURN_IN * (COST_PROXY + COST_HUMAN)
print(f"Burn-in set size           : {N_BURN_IN} conversations")
print(f"Burn-in total cost         : ${burn_in_cost:.2f}  ({N_BURN_IN} x ${COST_PROXY + COST_HUMAN:.2f} each)")
print()
print(f"Estimated Var(H)           : {sampler._y_true_variance:.4f}")
print(f"Estimated MSE(H, G)        : {sampler._mean_squared_error:.4f}")

## The optimal human annotation rate

With $\text{Var}(H)$ and $\text{MSE}(H, G)$ in hand, the sampler computes $\pi^*$, the Bernoulli probability at which each main-set conversation is sent to a human annotator:

$$\pi^* = \sqrt{\frac{c_g}{c_h} \cdot \frac{\text{MSE}(H, G)}{\text{Var}(H) - \text{MSE}(H, G)}}$$

Two intuitions from this formula:

- A **higher cost ratio** $c_h / c_g$ (expensive human relative to LLM) drives $\pi^*$ down: when humans are costly, it becomes efficient to rely more on the cheap proxy and send only a fraction to a human annotator.
- A **more accurate proxy** (small $\text{MSE}$) also drives $\pi^*$ down: when the LLM judge already tracks the truth closely, few human annotations are needed to correct the residual bias.

The expected cost per main-set conversation under $\pi^*$ is $c_g + c_h \cdot \pi^*$. Given the budget $B$, the sampler can cover at most $T = \lfloor B / (c_g + c_h \cdot \pi^*) \rfloor$ conversations.

In [ ]:
pi_optimal = sampler._compute_optimal_probability(COST_HUMAN, COST_PROXY)
cost_per_sample_optimal = COST_PROXY + COST_HUMAN * pi_optimal
T_max = int(np.floor(BUDGET_MAIN / cost_per_sample_optimal))

cost_per_sample_uniform = COST_PROXY + COST_HUMAN  # pi = 1
n_uniform_main = int(np.floor(BUDGET_MAIN / cost_per_sample_uniform))

print(f"Cost-optimal: pi* = {pi_optimal:.3f}")
print(f"  Cost per sample : ${cost_per_sample_optimal:.4f}")
print(f"  Conversations covered: T = {T_max:,}  (all {N_MAIN:,} fit within budget: {T_max >= N_MAIN})")
print()
print("Uniform (pi=1):")
print(f"  Cost per sample : ${cost_per_sample_uniform:.2f}")
print(f"  Conversations covered: {n_uniform_main}  (out of {N_MAIN:,})")
print()
print("Proxy labels available for PPI:")
print(f"  Cost-optimal: {N_BURN_IN + N_MAIN:,}  (burn-in + all {N_MAIN:,} main)")
print(f"  Uniform     : {N_BURN_IN + n_uniform_main}  (burn-in + {n_uniform_main} main)")

## Step 2: Apply cost-optimal sampling to the remaining conversations

With $\pi^*$ determined, the sampler draws independent Bernoulli indicators $\xi_i \sim \text{Bernoulli}(\pi^*)$ for each of the 2,100 main conversations. Conversations with $\xi_i = 1$ receive a human annotation alongside the LLM label; the rest receive only the LLM label.

The sampler returns three values:

| Return value | Meaning |
|---|---|
| `indices` | Sorted indices of the selected conversations (all N_MAIN when budget is sufficient) |
| `pi` | The optimal probability $\pi^*$ used for every Bernoulli draw |
| `xi` | Bernoulli indicators: `1` = human annotation obtained, `0` = proxy label only |

In [ ]:
indices, pi_sampled, xi = sampler.sample(
    n_samples=N_MAIN,
    y_true_cost=COST_HUMAN,
    y_proxy_cost=COST_PROXY,
    budget=BUDGET_MAIN,
    random_seed=RANDOM_SEED,
)

# Build y_true: burn-in always labeled; main conversations labeled only when xi = 1
y_true_optimal = np.full(N_TOTAL, np.nan)
y_true_optimal[:N_BURN_IN] = y_true_burn_in
main_annotated_indices = indices[xi == 1]
y_true_optimal[N_BURN_IN + main_annotated_indices] = y_true_oracle[N_BURN_IN + main_annotated_indices]

# Build pi array: burn-in always annotated (pi = 1); all main samples share pi = pi*
pi_array_optimal = np.full(N_TOTAL, pi_sampled)
pi_array_optimal[:N_BURN_IN] = 1.0

n_main_annotated = int(xi.sum())
n_total_annotated_optimal = N_BURN_IN + n_main_annotated

In [ ]:
print("Cost-optimal sampling results")
print(f"  Main conversations processed  : {len(indices):,}  (of {N_MAIN:,})")
print(f"  Main conversations with human : {n_main_annotated} ({n_main_annotated / len(indices):.1%})")
print(f"  Total human annotations       : {n_total_annotated_optimal}  ({N_BURN_IN} burn-in + {n_main_annotated} main)")
print(f"  Optimal rate (theoretical)    : pi* = {pi_sampled:.3f}")
print(f"  Realized annotation rate      : {n_main_annotated / len(indices):.3f}")

## Uniform sampler: the naive baseline with the same budget

A **uniform sampler** applies `np.random.choice` to pick which conversations to process, then fully annotates every selected conversation with a human ($\pi = 1$). With a budget of $400 and a per-conversation cost of $1.01 (LLM + human), this covers only **396 main conversations** — leaving 1,704 conversations with no proxy label at all.

Because every processed conversation already has a human label, there is no proxy-only unlabeled set for prediction-powered inference to leverage. The estimator therefore falls back to a classical sample mean over the 496 human-labeled conversations (burn-in + 396 main).

The cost-optimal sampler makes a fundamentally different choice: by setting $\pi^* \approx 18\%$, it **covers all 2,100 conversations** with a proxy label while sending only a fraction to a human annotator. This creates the labeled/unlabeled split that PPI requires, allowing all proxy labels to contribute to variance reduction.

In [ ]:
# Uniform sampler: select n_uniform_main random main conversations, fully annotate each (pi = 1)
rng_uniform = np.random.default_rng(RANDOM_SEED + 1)
uniform_main_indices = np.sort(rng_uniform.choice(N_MAIN, size=n_uniform_main, replace=False))

# Human-labeled samples: burn-in + selected main (all selected are fully annotated with pi=1)
y_human_uniform = np.hstack(
    [
        y_true_burn_in,
        y_true_oracle[N_BURN_IN + uniform_main_indices],
    ]
)
n_total_annotated_uniform = N_BURN_IN + n_uniform_main

print("Uniform sampling results")
print(f"  Main conversations processed  : {n_uniform_main}  (of {N_MAIN:,})")
print("  All processed have human label: True (pi = 1)")
print(f"  Total human annotations       : {n_total_annotated_uniform}  ({N_BURN_IN} burn-in + {n_uniform_main} main)")
print(f"  Proxy labels for PPI          : {n_total_annotated_uniform}  (only processed samples)")
print()
print("  => No unlabeled-but-proxied samples available: PPI cannot be applied")

## Estimating the hallucination rate under each strategy

With the samples drawn, we can now estimate the hallucination rate for each strategy:

- **LLM judge only**: average over all 2,200 proxy labels — cheap, but biased.
- **Uniform sampler**: classical sample mean over the 496 human-labeled conversations — unbiased, but the 1,704 unprocessed conversations and their proxy labels are wasted.
- **Cost-optimal PPI**: `ASIMeanEstimator` combines IPW (to handle the non-uniform rates across burn-in and main) with PPI++ (to leverage all 2,200 proxy labels for variance reduction).

In [ ]:
# LLM judge: mean over all 2,200 proxy labels
judge_estimate = ClassicalMeanEstimator().estimate(y_proxy)
judge_mean = judge_estimate.mean
judge_lower = judge_estimate.confidence_interval.lower_bound
judge_upper = judge_estimate.confidence_interval.upper_bound

# Uniform sampler: classical mean over 496 human-labeled conversations
uniform_estimate = ClassicalMeanEstimator().estimate(y_human_uniform)
uniform_mean = uniform_estimate.mean
uniform_lower = uniform_estimate.confidence_interval.lower_bound
uniform_upper = uniform_estimate.confidence_interval.upper_bound

# Cost-optimal PPI: ASI on all 2,200 samples (proxy) + ~470 human-labeled
ppi_result = ASIMeanEstimator().estimate(
    y_true_optimal,
    y_proxy,
    pi_array_optimal,
    metric_name="Hallucination Rate",
    confidence_level=0.95,
)

In [ ]:
sep = "-" * 76
print(sep)
print(f"{'Method':<40} {'Estimate':>8}   {'95% confidence interval':>16}")
print(sep)
print(f"{'LLM Judge only  (2,200 proxy)':<40} {judge_mean:>7.1%}   [{judge_lower:.1%}, {judge_upper:.1%}]")
print(
    f"{'Uniform sampler (n=' + str(n_total_annotated_uniform) + ' human, no PPI)':<40} "
    f"{uniform_mean:>7.1%}   [{uniform_lower:.1%}, {uniform_upper:.1%}]"
)
print(
    f"{'Cost-optimal PPI (n=' + str(n_total_annotated_optimal) + ' human + 2,200 proxy)':<40} "
    f"{ppi_result.mean:>7.1%}   "
    f"[{ppi_result.confidence_interval.lower_bound:.1%}, "
    f"{ppi_result.confidence_interval.upper_bound:.1%}]"
)
print(sep)
print(f"{'True rate  (simulation)':<40} {'10.0%':>8}")

In [ ]:
print(ppi_result.summary())

## Cost-optimal PPI is sharper than the uniform sampler

The plot below compares **point estimates** and **95% confidence intervals** for all three methods against the true hallucination rate (dashed line):

- **LLM judge**: very narrow interval, but wrong — the 100x cheaper labels come with a bias that invalidates the estimate.
- **Uniform sampler**: unbiased, but the $400 budget buys only 396 processed conversations. There is no proxy-only unlabeled set, so PPI cannot be applied and the confidence interval is wide.
- **Cost-optimal PPI**: unbiased *and* narrower — the same budget covers all 2,100 conversations with a proxy label, creates the labeled/unlabeled split PPI needs, and applies a bias correction on top. The effective sample size reported in the summary above shows by how much.

In [ ]:
estimates = [
    (
        f"LLM Judge\n({N_TOTAL:,}  |  raw proxy)",
        judge_mean,
        judge_lower,
        judge_upper,
        C_JUDGE,
    ),
    (
        f"Uniform Sampler\n({n_total_annotated_uniform}  |  pi=1, no PPI)",
        uniform_mean,
        uniform_lower,
        uniform_upper,
        C_UNIFORM,
    ),
    (
        f"Cost-Optimal PPI (GLIDE)\n({n_total_annotated_optimal} human  +  {N_TOTAL:,} proxy)\n"
        f"(pi* = {pi_sampled:.2f})",
        ppi_result.mean,
        ppi_result.confidence_interval.lower_bound,
        ppi_result.confidence_interval.upper_bound,
        C_GLIDE,
    ),
]

fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = [2, 1, 0]

for y, (label, mean, lo, hi, color) in zip(y_pos, estimates):
    ax.plot([lo, hi], [y, y], color=color, linewidth=4, solid_capstyle="round", zorder=3)
    for xc in [lo, hi]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2], color=color, linewidth=2.5, zorder=3)
    ax.scatter(mean, y, s=200, color=color, zorder=5, edgecolors="white", linewidths=2.5)
    ax.text(mean, y + 0.34, f"{mean:.1%}", ha="center", va="bottom", fontsize=12, color=color, fontweight="bold")
    ax.text(mean, y - 0.34, f"[{lo:.1%},  {hi:.1%}]", ha="center", va="top", fontsize=11, color="#888888")

ax.axvline(TRUE_RATE, color=C_TRUTH, linestyle="--", linewidth=2.5, zorder=4)
ax.text(TRUE_RATE + 0.004, 2.72, "True rate  10%", color=C_TRUTH, fontsize=10.5, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in estimates], fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Estimated Hallucination Rate", fontsize=12)
ax.set_title(
    "Cost-Optimal PPI Delivers a Sharper Estimate Than Uniform Sampling",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlim(-0.01, 0.26)
ax.set_ylim(-0.8, 3.2)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Testing whether the hallucination rate exceeds the threshold

GLIDE's output enables a formal hypothesis test: is the true hallucination rate significantly higher than 5%?

$$H_0 : \mu = 5\% \qquad H_1 : \mu > 5\%$$

The LLM judge's interval sits close to 5%, making rejection unlikely. The uniform sampler is unbiased but its wide interval leads to a conservative decision. Cost-optimal PPI uses the **same budget** but additionally leverages all 2,200 proxy labels, narrowing the interval enough to yield a decisive test.

In [ ]:
z_stat, p_value, _ = ppi_result.confidence_interval.test_null_hypothesis(
    h0_value=0.05,
    alternative="larger",
)

sep = "-" * 48
print("Hypothesis test — Cost-Optimal PPI (GLIDE)")
print(sep)
print("H0 : hallucination rate = 5%   (LLM says so)")
print("H1 : hallucination rate > 5%   (users complain!)")
print()
print(f"z-statistic : {z_stat:.2f}")
print(f"p-value     : {p_value:.10f}")
print()
if p_value < 0.05:
    print("Decision  : Given the p-value is lower than our threshold (p-value < 0.05), we reject H0 at the 5% level.")
    print("=> The true hallucination rate is significantly above 5%.")
else:
    print(
        "Decision  : Given the p-value is higher than our threshold (p-value >= 0.05), "
        "we cannot reject H0 at the 5% level."
    )

The null hypothesis is rejected, signalling that the hallucination rate is significantly above the fixed threshold.

Let us run the same hypothesis test using the uniform sampler estimate.

In [ ]:
z_stat_u, p_value_u, _ = uniform_estimate.confidence_interval.test_null_hypothesis(
    h0_value=0.05,
    alternative="larger",
)

sep = "-" * 48
print("Hypothesis test — Uniform Sampler (no PPI)")
print(sep)
print("H0 : hallucination rate = 5%   (LLM says so)")
print("H1 : hallucination rate > 5%   (users complain!)")
print()
print(f"z-statistic : {z_stat_u:.2f}")
print(f"p-value     : {p_value_u:.10f}")
print()
if p_value_u < 0.05:
    print("Decision  : Given the p-value is lower than our threshold (p-value < 0.05), we reject H0 at the 5% level.")
    print("=> The true hallucination rate is significantly above 5%.")
else:
    print(
        "Decision  : Given the p-value is higher than our threshold (p-value >= 0.05), "
        "we cannot reject H0 at the 5% level."
    )

The null hypothesis is not rejected: spending the full $400 budget on fully-annotated conversations (uniform, $\pi = 1$) leaves too few labeled samples to draw a firm statistical conclusion. By contrast, the cost-optimal sampler allocates the same budget to cover all 2,100 conversations with proxy labels and annotates a fraction at the optimal rate $\pi^*$, giving prediction-powered inference enough unlabeled-but-proxied samples to shrink the confidence interval decisively.

## Summary: what each layer adds

| | LLM Judge | Uniform Sampler | **Cost-Optimal PPI** |
|-|-----------|-----------------|---------------------|
| Proxy labels used | 2,200 | 496 | 2,200 |
| Human labels | 0 | 496 | ~470 |
| Annotation rate $\pi$ | — | 1 (all processed) | ~0.18 (optimal) |
| Unbiased estimate | ❌ | ✅ | ✅ |
| PPI applicable | ❌ | ❌ | ✅ |
| Narrow CI | 🟠 *(misleading)* | ❌ | ✅ |

**Key takeaways:**

1. **LLM judges are biased.** A narrow confidence interval around the wrong value gives false confidence.

2. **Uniform sampling ($\pi = 1$) wastes the proxy labels.** Spending every dollar on full annotations leaves the 1,704 unprocessed conversations with no proxy label, blocking PPI from contributing variance reduction.

3. **The cost ratio determines the optimal annotation rate.** A 100x price gap means annotating only ~18% of conversations with humans is already optimal — the residual bias is corrected by PPI at a fraction of the cost.

4. **Coverage beats density.** The cost-optimal sampler trades a few human annotations for full proxy coverage, enabling PPI to leverage all 2,200 cheap labels. Same budget, tighter interval.

5. **The burn-in calibration pays for itself.** Annotating 100 conversations fully upfront pins down the cost-optimal rate; every subsequent annotation is spent at the statistically optimal fraction.

---

*Want to go further? The [ASI tutorial](../asi/) shows how to also exploit judge uncertainty scores to concentrate the annotation budget on the hardest conversations, adding an active-sampling layer on top of cost optimisation.*